# Phase 1 Baseline Validation - All Datasets

Comprehensive reproducibility study across **CIFAR-10**, **Fashion-MNIST**, and **CIFAR-100** datasets.

For each dataset, run 5 experiments to establish baseline statistics:
- **3-seed baseline**: AdamW, lr=0.001, seeds=[42, 123, 456]
- **Higher LR variant**: lr=0.003, seed=42
- **SGD baseline**: SGD, lr=0.01, momentum=0.9, seed=42

**Total experiments**: 15 (5 per dataset × 3 datasets)

Includes test set evaluation, timing metrics, and saves individual + combined results:
- `phase1_cifar10_results.json`
- `phase1_fashion_mnist_results.json`
- `phase1_cifar100_results.json`
- `phase1_all_datasets_results.json` (combined)

In [5]:
# Setup and imports
import sys
import json
import time
from pathlib import Path

candidate_roots = [
    Path('/workspaces/ouroboros'),
    Path('/content/drive/MyDrive/ouroboros'),
    Path('/content/ouroboros'),
    Path('.').resolve().parent,  # Local development
]
project_root = next((p for p in candidate_roots if (p / 'src').exists()), None)
if project_root is None:
    raise FileNotFoundError('Project root not found. Update candidate_roots.')
sys.path.insert(0, str(project_root))

import torch
from torch import nn
from torch.optim import AdamW, SGD
from tqdm.auto import tqdm

from src.data_loaders import (
    get_cifar10_loaders, get_cifar10_test_loader,
    get_fashion_mnist_loaders, get_fashion_mnist_test_loader,
    get_cifar100_loaders, get_cifar100_test_loader,
)
from src.models import CNN3Layer, DEFAULT_CHANNELS, WIDE_CHANNELS, count_parameters
from src.trainer import train_epoch, validate_epoch, get_epoch_scheduler
from src.utils import set_seed, get_device, ensure_dirs

device = get_device()
ensure_dirs('results/phase1', 'checkpoints')
print(f"Device: {device}")
print(f"Project root: {project_root}")


Device: cuda
Project root: /content/drive/MyDrive/ouroboros


## Define Experiment Runner

In [6]:
def run_experiment(config: dict) -> dict:
    """Run a single experiment with config dict. Returns results with test accuracy."""
    set_seed(config["seed"], deterministic=False)

    # Dataset selection
    dataset = config["dataset"]
    if dataset == "cifar10":
        train_loader, val_loader = get_cifar10_loaders(config["batch_size"], config["num_workers"], config["data_dir"], seed=config["seed"])
        test_loader = get_cifar10_test_loader(config["batch_size"], config["num_workers"], config["data_dir"])
        num_classes = 10
        in_channels = 3
        channels = DEFAULT_CHANNELS
    elif dataset == "fashion_mnist":
        train_loader, val_loader = get_fashion_mnist_loaders(config["batch_size"], config["num_workers"], config["data_dir"], seed=config["seed"])
        test_loader = get_fashion_mnist_test_loader(config["batch_size"], config["num_workers"], config["data_dir"])
        num_classes = 10
        in_channels = 1
        channels = DEFAULT_CHANNELS
    elif dataset == "cifar100":
        train_loader, val_loader = get_cifar100_loaders(config["batch_size"], config["num_workers"], config["data_dir"], seed=config["seed"])
        test_loader = get_cifar100_test_loader(config["batch_size"], config["num_workers"], config["data_dir"])
        num_classes = 100
        in_channels = 3
        channels = WIDE_CHANNELS  # Use wider architecture for CIFAR-100
    else:
        raise ValueError(f"Unknown dataset: {dataset}")

    model = CNN3Layer(num_classes=num_classes, in_channels=in_channels, channels=channels).to(device)

    if config["optimizer"] == "sgd":
        optimizer = SGD(model.parameters(), lr=config["lr"], momentum=config.get("momentum", 0.9), weight_decay=config["weight_decay"])
    else:
        optimizer = AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])

    criterion = nn.CrossEntropyLoss()
    scheduler = get_epoch_scheduler(optimizer, config["epochs"], config["warmup_epochs"])

    print(f"[DATASET] {dataset.upper()} | classes={num_classes} | channels={in_channels}")
    print(f"[OPTIMIZER] {type(optimizer).__name__} | lr={config['lr']} | wd={config['weight_decay']}")
    print(f"[MODEL] params={count_parameters(model):,}")

    # NVIDIA Best Practice: Warm-up iteration
    if device.type == "cuda":
        warmup_batch = next(iter(train_loader))
        with torch.no_grad():
            _ = model(warmup_batch[0].to(device))
        torch.cuda.synchronize()

    epoch_times, train_history, val_history = [], [], []

    # NVIDIA Best Practice: GPU sync for accurate timing
    if device.type == "cuda":
        torch.cuda.synchronize()
    total_start = time.perf_counter()

    for epoch in tqdm(range(1, config["epochs"] + 1), desc=config["name"], unit="epoch"):
        if device.type == "cuda":
            torch.cuda.synchronize()
        epoch_start = time.perf_counter()

        train_metrics = train_epoch(model, train_loader, optimizer, criterion, device, amp_enabled=(device.type == "cuda"))
        val_metrics = validate_epoch(model, val_loader, criterion, device)
        scheduler.step()

        if device.type == "cuda":
            torch.cuda.synchronize()
        epoch_times.append(time.perf_counter() - epoch_start)
        train_history.append(train_metrics)
        val_history.append(val_metrics)

    if device.type == "cuda":
        torch.cuda.synchronize()
    total_time = time.perf_counter() - total_start

    test_metrics = validate_epoch(model, test_loader, criterion, device)

    # Memory cleanup
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return {
        "experiment_name": config["name"],
        "config": {k: v for k, v in config.items() if k != "name"},
        "final_train_acc": train_history[-1]["accuracy"],
        "final_val_acc": val_history[-1]["accuracy"],
        "test_acc": test_metrics["accuracy"],
        "test_loss": test_metrics["loss"],
        "total_time_sec": total_time,
        "mean_epoch_time": sum(epoch_times) / len(epoch_times),
        "params": count_parameters(model),
    }


## Run 5 Experiments

In [7]:
# Default config
DEFAULT_CONFIG = {
    "epochs": 5,
    "batch_size": 128,
    "lr": 0.001,
    "weight_decay": 1e-4,
    "warmup_epochs": 1,
    "optimizer": "adamw",
    "momentum": 0.9,
    "num_workers": 2,
    "data_dir": "assets",
}

# Define experiments for each dataset
datasets = ["cifar10", "fashion_mnist", "cifar100"]
all_dataset_results = {}

for dataset in datasets:
    print(f"\n{'#'*70}")
    print(f"# STARTING EXPERIMENTS FOR: {dataset.upper()}")
    print(f"{'#'*70}\n")

    # 5 experiments per dataset: 3-seed baseline + 2 variants
    experiments = [
        {"name": f"{dataset}_seed42", **DEFAULT_CONFIG, "dataset": dataset, "seed": 42},
        {"name": f"{dataset}_seed123", **DEFAULT_CONFIG, "dataset": dataset, "seed": 123},
        {"name": f"{dataset}_seed456", **DEFAULT_CONFIG, "dataset": dataset, "seed": 456},
        {"name": f"{dataset}_higher_lr", **DEFAULT_CONFIG, "dataset": dataset, "seed": 42, "lr": 0.003},
        {"name": f"{dataset}_sgd", **DEFAULT_CONFIG, "dataset": dataset, "seed": 42, "optimizer": "sgd", "lr": 0.01},
    ]

    # Run all experiments for this dataset
    dataset_results = []
    for exp in experiments:
        print(f"\n{'='*70}")
        print(f"EXPERIMENT: {exp['name']}")
        print(f"{'='*70}")

        result = run_experiment(exp)
        dataset_results.append(result)

        print(f"\n[RESULTS]")
        print(f"  Train Acc: {result['final_train_acc']:.4f} ({result['final_train_acc']*100:.2f}%)")
        print(f"  Val Acc:   {result['final_val_acc']:.4f} ({result['final_val_acc']*100:.2f}%)")
        print(f"  Test Acc:  {result['test_acc']:.4f} ({result['test_acc']*100:.2f}%)")
        print(f"  Test Loss: {result['test_loss']:.4f}")
        print(f"  Total Time: {result['total_time_sec']:.1f}s")
        print(f"  Avg Epoch Time: {result['mean_epoch_time']:.2f}s")

    all_dataset_results[dataset] = dataset_results

    print(f"\n{'#'*70}")
    print(f"# COMPLETED: {dataset.upper()}")
    print(f"{'#'*70}\n")



######################################################################
# STARTING EXPERIMENTS FOR: CIFAR10
######################################################################


EXPERIMENT: cifar10_seed42


100%|██████████| 170M/170M [00:04<00:00, 38.5MB/s]


[DATASET] CIFAR10 | classes=10 | channels=3
[OPTIMIZER] AdamW | lr=0.001 | wd=0.0001
[MODEL] params=94,986


cifar10_seed42:   0%|          | 0/5 [00:00<?, ?epoch/s]


[RESULTS]
  Train Acc: 0.6487 (64.87%)
  Val Acc:   0.6386 (63.86%)
  Test Acc:  0.6376 (63.76%)
  Test Loss: 1.0140
  Total Time: 91.5s
  Avg Epoch Time: 18.30s

EXPERIMENT: cifar10_seed123
[DATASET] CIFAR10 | classes=10 | channels=3
[OPTIMIZER] AdamW | lr=0.001 | wd=0.0001
[MODEL] params=94,986


...


[RESULTS]
  Train Acc: 0.3549 (35.49%)
  Val Acc:   0.3326 (33.26%)
  Test Acc:  0.3474 (34.74%)
  Test Loss: 2.5760
  Total Time: 89.2s
  Avg Epoch Time: 17.84s

EXPERIMENT: cifar100_higher_lr
[DATASET] CIFAR100 | classes=100 | channels=3
[OPTIMIZER] AdamW | lr=0.003 | wd=0.0001
[MODEL] params=397,412


cifar100_higher_lr:   0%|          | 0/5 [00:00<?, ?epoch/s]


[RESULTS]
  Train Acc: 0.4085 (40.85%)
  Val Acc:   0.3908 (39.08%)
  Test Acc:  0.3966 (39.66%)
  Test Loss: 2.3499
  Total Time: 90.3s
  Avg Epoch Time: 18.06s

EXPERIMENT: cifar100_sgd
[DATASET] CIFAR100 | classes=100 | channels=3
[OPTIMIZER] SGD | lr=0.01 | wd=0.0001
[MODEL] params=397,412


cifar100_sgd:   0%|          | 0/5 [00:00<?, ?epoch/s]


[RESULTS]
  Train Acc: 0.2626 (26.26%)
  Val Acc:   0.2504 (25.04%)
  Test Acc:  0.2638 (26.38%)
  Test Loss: 2.9845
  Total Time: 91.0s
  Avg Epoch Time: 18.20s

######################################################################
# COMPLETED: CIFAR100
######################################################################



## Compute Baseline Statistics & Save Results

In [8]:
# Compute baseline statistics for each dataset
def mean(vals): return sum(vals) / len(vals)
def std(vals):
    m = mean(vals)
    return (sum((x - m)**2 for x in vals) / len(vals)) ** 0.5

all_stats = {}

for dataset in datasets:
    dataset_results = all_dataset_results[dataset]
    baseline_results = dataset_results[:3]  # First 3 experiments are 3-seed baseline
    baseline_val_accs = [r["final_val_acc"] for r in baseline_results]
    baseline_test_accs = [r["test_acc"] for r in baseline_results]

    stats = {
        "baseline_val_mean": mean(baseline_val_accs),
        "baseline_val_std": std(baseline_val_accs),
        "baseline_test_mean": mean(baseline_test_accs),
        "baseline_test_std": std(baseline_test_accs),
    }
    all_stats[dataset] = stats

    # Save individual dataset results
    output = {
        "dataset": dataset,
        "experiments": dataset_results,
        "baseline_statistics": stats,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }

    output_path = Path(f"results/phase1/phase1_{dataset}_results.json")
    with open(output_path, "w") as f:
        json.dump(output, f, indent=2)

    print(f"\n{'='*70}")
    print(f"BASELINE STATISTICS - {dataset.upper()} (3-SEED AVERAGE)")
    print(f"{'='*70}")
    print(f"Seeds: {[r['config']['seed'] for r in baseline_results]}")
    print(f"\nValidation Accuracy: {stats['baseline_val_mean']:.4f} ± {stats['baseline_val_std']:.4f}")
    print(f"                     ({stats['baseline_val_mean']*100:.2f}% ± {stats['baseline_val_std']*100:.2f}%)")
    print(f"\nTest Accuracy:       {stats['baseline_test_mean']:.4f} ± {stats['baseline_test_std']:.4f}")
    print(f"                     ({stats['baseline_test_mean']*100:.2f}% ± {stats['baseline_test_std']*100:.2f}%)")
    print(f"\nResults saved to: {output_path}")
    print(f"{'='*70}")

# Save combined results
combined_output = {
    "datasets": list(all_dataset_results.keys()),
    "all_experiments": all_dataset_results,
    "all_statistics": all_stats,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
}

combined_path = Path("results/phase1/phase1_all_datasets_results.json")
with open(combined_path, "w") as f:
    json.dump(combined_output, f, indent=2)

print(f"\n{'='*70}")
print(f"Combined results saved to: {combined_path}")
print(f"{'='*70}")



BASELINE STATISTICS - CIFAR10 (3-SEED AVERAGE)
Seeds: [42, 123, 456]

Validation Accuracy: 0.6403 ± 0.0050
                     (64.03% ± 0.50%)

Test Accuracy:       0.6414 ± 0.0068
                     (64.14% ± 0.68%)

Results saved to: results/phase1/phase1_cifar10_results.json

BASELINE STATISTICS - FASHION_MNIST (3-SEED AVERAGE)
Seeds: [42, 123, 456]

Validation Accuracy: 0.8704 ± 0.0040
                     (87.04% ± 0.40%)

Test Accuracy:       0.8632 ± 0.0040
                     (86.32% ± 0.40%)

Results saved to: results/phase1/phase1_fashion_mnist_results.json

BASELINE STATISTICS - CIFAR100 (3-SEED AVERAGE)
Seeds: [42, 123, 456]

Validation Accuracy: 0.3411 ± 0.0063
                     (34.11% ± 0.63%)

Test Accuracy:       0.3493 ± 0.0014
                     (34.93% ± 0.14%)

Results saved to: results/phase1/phase1_cifar100_results.json

Combined results saved to: results/phase1/phase1_all_datasets_results.json


## Results Summary Table

In [9]:
# Display results table for each dataset
for dataset in datasets:
    dataset_results = all_dataset_results[dataset]

    print(f"\n{'='*70}")
    print(f"DETAILED RESULTS TABLE - {dataset.upper()}")
    print(f"{'='*70}")
    print(f"{'Experiment':<25} {'Optimizer':<8} {'LR':<8} {'Val Acc':<10} {'Test Acc':<10} {'Time (s)':<10}")
    print("-" * 70)
    for r in dataset_results:
        cfg = r["config"]
        print(f"{r['experiment_name']:<25} {cfg['optimizer']:<8} {cfg['lr']:<8.4f} {r['final_val_acc']:<10.4f} {r['test_acc']:<10.4f} {r['total_time_sec']:<10.1f}")
    print("=" * 70)



DETAILED RESULTS TABLE - CIFAR10
Experiment                Optimizer LR       Val Acc    Test Acc   Time (s)  
----------------------------------------------------------------------
cifar10_seed42            adamw    0.0010   0.6386     0.6376     91.5      
cifar10_seed123           adamw    0.0010   0.6352     0.6356     88.3      
cifar10_seed456           adamw    0.0010   0.6470     0.6509     88.3      
cifar10_higher_lr         adamw    0.0030   0.6862     0.6851     87.0      
cifar10_sgd               sgd      0.0100   0.6204     0.6099     86.2      

DETAILED RESULTS TABLE - FASHION_MNIST
Experiment                Optimizer LR       Val Acc    Test Acc   Time (s)  
----------------------------------------------------------------------
fashion_mnist_seed42      adamw    0.0010   0.8690     0.8646     94.5      
fashion_mnist_seed123     adamw    0.0010   0.8663     0.8578     95.4      
fashion_mnist_seed456     adamw    0.0010   0.8758     0.8672     92.3      
fashion_mnis

In [10]:
# Final Comprehensive Summary
print(f"\n{'='*70}")
print("FINAL SUMMARY - Phase 1 Reproducibility Study")
print(f"{'='*70}")
print(f"Total Datasets Tested: {len(datasets)}")
print(f"Datasets: {', '.join([d.upper() for d in datasets])}")
print(f"Experiments per Dataset: 5 (3-seed baseline + 2 variants)")
print(f"Total Experiments: {sum(len(all_dataset_results[d]) for d in datasets)}")

# Summary for each dataset
for dataset in datasets:
    dataset_results = all_dataset_results[dataset]
    baseline_results = dataset_results[:3]
    stats = all_stats[dataset]

    print(f"\n{'─'*70}")
    print(f"{dataset.upper()} BASELINE PERFORMANCE")
    print(f"{'─'*70}")
    print(f"Model Architecture: CNN3Layer")
    print(f"Model Parameters: {dataset_results[0]['params']:,}")
    print(f"Training Epochs: 5 | Batch Size: 128")

    print(f"\n3-Seed Baseline (AdamW, lr=0.001):")
    for i, r in enumerate(baseline_results, 1):
        print(f"  Seed {r['config']['seed']:>3}: Val={r['final_val_acc']:.4f} | Test={r['test_acc']:.4f}")
    print(f"\n  Average:  Val={stats['baseline_val_mean']:.4f}±{stats['baseline_val_std']:.4f} | Test={stats['baseline_test_mean']:.4f}±{stats['baseline_test_std']:.4f}")
    print(f"  Reproducibility (std): ±{stats['baseline_test_std']:.4f} ({stats['baseline_test_std']*100:.2f}%)")

    print(f"\nVariant Experiments:")
    for r in dataset_results[3:]:
        cfg = r["config"]
        variant_type = "Higher LR" if "higher_lr" in r['experiment_name'] else "SGD"
        print(f"  {variant_type:<12}: {cfg['optimizer']}, lr={cfg['lr']:.4f} | Test={r['test_acc']:.4f}")

# Cross-dataset comparison
print(f"\n{'─'*70}")
print("CROSS-DATASET COMPARISON (BASELINE AVERAGES)")
print(f"{'─'*70}")
print(f"{'Dataset':<20} {'Test Acc':<15} {'Std Dev':<15} {'Params':<15}")
print("-" * 70)
for dataset in datasets:
    stats = all_stats[dataset]
    params = all_dataset_results[dataset][0]['params']
    print(f"{dataset.upper():<20} {stats['baseline_test_mean']:.4f}          ±{stats['baseline_test_std']:.4f}         {params:,}")

# Key insights
print(f"\n{'─'*70}")
print("KEY INSIGHTS")
print(f"{'─'*70}")

best_by_dataset = {}
for dataset in datasets:
    dataset_results = all_dataset_results[dataset]
    best = max(dataset_results, key=lambda x: x['test_acc'])
    worst = min(dataset_results, key=lambda x: x['test_acc'])
    best_by_dataset[dataset] = best

    print(f"\n{dataset.upper()}:")
    print(f"  Best Configuration:  {best['experiment_name']} (Test Acc: {best['test_acc']:.4f})")
    print(f"  Worst Configuration: {worst['experiment_name']} (Test Acc: {worst['test_acc']:.4f})")
    print(f"  Performance Range:   {(best['test_acc'] - worst['test_acc']):.4f} ({(best['test_acc'] - worst['test_acc'])*100:.2f}%)")



FINAL SUMMARY - Phase 1 Reproducibility Study
Total Datasets Tested: 3
Datasets: CIFAR10, FASHION_MNIST, CIFAR100
Experiments per Dataset: 5 (3-seed baseline + 2 variants)
Total Experiments: 15

──────────────────────────────────────────────────────────────────────
CIFAR10 BASELINE PERFORMANCE
──────────────────────────────────────────────────────────────────────
Model Architecture: CNN3Layer
Model Parameters: 94,986
Training Epochs: 5 | Batch Size: 128

3-Seed Baseline (AdamW, lr=0.001):
  Seed  42: Val=0.6386 | Test=0.6376
  Seed 123: Val=0.6352 | Test=0.6356
  Seed 456: Val=0.6470 | Test=0.6509

  Average:  Val=0.6403±0.0050 | Test=0.6414±0.0068
  Reproducibility (std): ±0.0068 (0.68%)

Variant Experiments:
  Higher LR   : adamw, lr=0.0030 | Test=0.6851
  SGD         : sgd, lr=0.0100 | Test=0.6099

──────────────────────────────────────────────────────────────────────
FASHION_MNIST BASELINE PERFORMANCE
──────────────────────────────────────────────────────────────────────
Model Arc